# Exploratory Data Analysis

Purpose : first pass at assessing experimental data on a subject and group basis.  
Date : Monday April 13, 2026


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

MIN_TRIALS = 5

df_clean = pd.read_csv('../data/processed/clean_data.csv')

trials = df_clean[df_clean['watch_only'] == 0].copy()
trials['pressed'] = trials['pressed'].astype('int64')
trials['outcome'] = trials['outcome'].astype('int64')
trials = trials.rename(columns={'prop_SC': 'sc_prop', 'prop_OC': 'oc_prop', 'prop_R': 'r_prop'})

_t1 = df_clean[df_clean['task'] == 1].groupby('subject_id').first().reset_index()
_press = trials.groupby('subject_id')['pressed'].mean().rename('overall_press_rate')
subject_meta = (
    _t1[['subject_id', 'group', 'prop_SC', 'prop_OC', 'prop_R', 'completion_min']]
    .rename(columns={'prop_SC': 'sc_prop', 'prop_OC': 'oc_prop', 'prop_R': 'r_prop'})
    .merge(_press, on='subject_id')
)

press_rates = (
    trials
    .groupby(['subject_id', 'group', 'sc_prop', 'task', 'block', 'trial_role'])
    .agg(
        n_trials  =('pressed', 'count'),
        n_pressed =('pressed', 'sum'),
        press_rate=('pressed', 'mean'),
        mean_rt   =('rt',      'mean'),
    )
    .reset_index()
)

learning_curves = (
    trials
    .groupby(['task', 'block', 'trial_n', 'trial_role'])['pressed']
    .mean()
    .reset_index()
    .rename(columns={'pressed': 'mean_press_rate'})
)

pr = press_rates[press_rates['n_trials'] >= MIN_TRIALS].copy()

print(f'trials:          {trials.shape}')
print(f'subject_meta:    {subject_meta.shape}')
print(f'press_rates:     {press_rates.shape}')
print(f'learning_curves: {learning_curves.shape}')


trials:          (11200, 20)
subject_meta:    (70, 7)
press_rates:     (828, 10)
learning_curves: (480, 5)


## 1 — Trial exposure and data quality

- Trial counts per subject per role per block per task
- SC property value distribution across subjects
- Group A vs B trial distribution comparison
- Confirm everyone got correct trial counts globally


In [2]:
# ── Global trial count sanity check
print(f"Active trials : {len(trials):,}  (expect {70*160:,} approx — watch_only excluded)")
print(f"Subjects      : {trials['subject_id'].nunique()}")
print()

# ── Trial counts per subject × role × block 
count_stats = (
    press_rates
    .groupby('trial_role')['n_trials']
    .describe()[['min', '25%', '50%', '75%', 'max']]
    .astype(int)
)
print("Trials per subject per role per block:")
print(count_stats)

low = press_rates[press_rates['n_trials'] < MIN_TRIALS]
print(f"\nCells below MIN_TRIALS={MIN_TRIALS}: {len(low)} / {len(press_rates)} ({100*len(low)/len(press_rates):.1f}%)")
print("\nLow-count cells by block and role:")
print(low.groupby(['block', 'trial_role']).size().to_string())
print()

# ── Group × SC property crosstab 
print("Subjects by group × SC property:")
print(pd.crosstab(subject_meta['group'], subject_meta['sc_prop'], margins=True))


Active trials : 11,200  (expect 11,200 approx — watch_only excluded)
Subjects      : 70

Trials per subject per role per block:
            min  25%  50%  75%  max
trial_role                         
OC            1    3    5    9   16
R             1    3    6    9   17
SC            8   14   26   42   51

Cells below MIN_TRIALS=5: 237 / 828 (28.6%)

Low-count cells by block and role:
block  trial_role
2      OC             11
       R               5
3      OC            112
       R             109

Subjects by group × SC property:
sc_prop  colour  size  velocity  All
group                               
A            12     9        18   39
B            10    16         5   31
All          22    25        23   70


## 2 — Learning signal

- Rolling mean SC press rate over block 2 trials (learning curve shape)
- SC outcome rate conditional on pressing — proxy for accuracy
- Average press rate per role per block, separately per task, separately per group
- SC vs OC rate relationship: do they move proportionately, or does SC selectively diverge? Plot distributions on same axes.


In [3]:
# 2.1: SC learning curve — rolling mean over block 2
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, task in zip(axes, [1, 2]):
    sc = (
        learning_curves
        .query("task == @task and block == 2 and trial_role == 'SC'")
        .sort_values('trial_n')
    )
    ax.plot(sc['trial_n'], sc['mean_press_rate'], alpha=0.3, color='steelblue', label='raw')
    ax.plot(sc['trial_n'], sc['mean_press_rate'].rolling(5, center=True).mean(),
            color='steelblue', linewidth=2, label='5-trial rolling mean')
    ax.set(title=f'Task {task} — SC learning curve (block 2)',
           xlabel='Trial', ylabel='Mean press rate')
    ax.legend()
plt.tight_layout()
plt.savefig('../output/g2_sc_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

# 2.2: SC outcome rate conditional on pressing
sc_pressed = trials[(trials['trial_role'] == 'SC') & (trials['pressed'] == 1)]
outcome_by_block = (
    sc_pressed
    .groupby(['task', 'block'])['outcome']
    .agg(mean_outcome='mean', n='count')
    .reset_index()
)
print("SC outcome rate given pressed (design check — expect ~0.95):")
print(outcome_by_block.to_string(index=False))

# 2.3: Average press rate per role per block, by task and group
summary = (
    pr.groupby(['group', 'task', 'block', 'trial_role'])['press_rate']
    .mean()
    .reset_index()
)
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
for row, group in enumerate(['A', 'B']):
    for col, task in enumerate([1, 2]):
        ax = axes[row][col]
        subset = summary.query("group == @group and task == @task")
        for role, colour in [('SC', 'steelblue'), ('OC', 'darkorange'), ('R', 'grey')]:
            d = subset[subset['trial_role'] == role]
            ax.plot(d['block'], d['press_rate'], marker='o', color=colour, label=role)
        ax.set(title=f'Group {group}, Task {task}',
               xlabel='Block', ylabel='Mean press rate', xticks=[1, 2, 3])
        ax.legend()
plt.tight_layout()
plt.savefig('../output/g2_press_rate_by_role_block.png', dpi=150, bbox_inches='tight')
plt.show()

# 2.4: SC vs OC press rate distributions on same axes
b2 = pr[(pr['block'] == 2)]
sc_rates = b2[b2['trial_role'] == 'SC'].set_index(['subject_id', 'task'])['press_rate']
oc_rates = b2[b2['trial_role'] == 'OC'].set_index(['subject_id', 'task'])['press_rate']

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True, sharex=True)
for ax, task in zip(axes, [1, 2]):
    sc = sc_rates.xs(task, level='task')
    oc = oc_rates.xs(task, level='task')
    ax.hist(sc, bins=15, alpha=0.5, color='steelblue', label='SC')
    ax.hist(oc, bins=15, alpha=0.5, color='darkorange', label='OC')
    ax.axvline(sc.mean(), color='steelblue', linestyle='--', linewidth=1.5, label=f'SC mean={sc.mean():.2f}')
    ax.axvline(oc.mean(), color='darkorange', linestyle='--', linewidth=1.5, label=f'OC mean={oc.mean():.2f}')
    ax.set(title=f'Task {task} — block 2 press rate distributions',
           xlabel='Press rate', ylabel='Subjects')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../output/g2_sc_oc_distributions.png', dpi=150, bbox_inches='tight')


SC outcome rate given pressed (design check — expect ~0.95):
 task  block  mean_outcome    n
    1      2      0.947011 1472
    1      3      0.946535  505
    2      2      0.957232 1286
    2      3      0.962376  505


/var/folders/60/gw08bfps0cldlsh___74vr780000gn/T/ipykernel_90870/1934636986.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/60/gw08bfps0cldlsh___74vr780000gn/T/ipykernel_90870/1934636986.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## GROUP 3 — Transfer

- SC press rate: task 1 block 2 vs task 2 block 2 early trials
- OC-in-task1 press rate in task 2 (should drop to R level if transfer is property-specific not general)
- Transfer index distribution across subjects
- Value-specific vs property-type transfer: identify subjects where SC positive value is same vs different across tasks, compare transfer indices


In [4]:
# ── G3.1: SC block 2 — Task 1 vs Task 2, full block and early trials ─────────
sc_b2 = (
    pr.query("block == 2 and trial_role == 'SC'")
    [['subject_id', 'group', 'sc_prop', 'task', 'press_rate']]
    .pivot_table(index=['subject_id', 'group', 'sc_prop'], columns='task', values='press_rate')
    .rename(columns={1: 'task1', 2: 'task2'})
    .reset_index()
    .dropna()
)
print(f"G3.1 — SC block 2 press rate (N={len(sc_b2)})")
print(f"  Task 1:  mean={sc_b2['task1'].mean():.3f}  sd={sc_b2['task1'].std():.3f}")
print(f"  Task 2:  mean={sc_b2['task2'].mean():.3f}  sd={sc_b2['task2'].std():.3f}")
t, p = stats.ttest_rel(sc_b2['task2'], sc_b2['task1'])
print(f"  Paired t (T2 vs T1): t={t:.3f}, p={p:.4f}")

early = (
    trials[trials['block'] == 2]
    .sort_values(['subject_id', 'task', 'trial_n'])
    .groupby(['subject_id', 'task'])
    .head(10)
)
sc_early = (
    early[early['trial_role'] == 'SC']
    .groupby(['subject_id', 'task'])['pressed']
    .agg(n='count', press_rate='mean')
    .reset_index()
    .query("n >= 3")
    .pivot_table(index='subject_id', columns='task', values='press_rate')
    .rename(columns={1: 'task1_early', 2: 'task2_early'})
    .reset_index()
    .dropna()
)
print(f"\n  Early trials — first 10 of block 2 (N={len(sc_early)})")
print(f"  Task 1 early: mean={sc_early['task1_early'].mean():.3f}")
print(f"  Task 2 early: mean={sc_early['task2_early'].mean():.3f}")
t_e, p_e = stats.ttest_rel(sc_early['task2_early'], sc_early['task1_early'])
print(f"  Paired t: t={t_e:.3f}, p={p_e:.4f}")

# G3.2: Former OC property press rate in Task 2
former_oc = pr.query("task==2 and block==2 and trial_role=='R'")[['subject_id','press_rate']].rename(columns={'press_rate':'former_oc_as_r_t2'})
oc_t1     = pr.query("task==1 and block==2 and trial_role=='OC'")[['subject_id','press_rate']].rename(columns={'press_rate':'oc_t1'})
r_t1      = pr.query("task==1 and block==2 and trial_role=='R'")[['subject_id','press_rate']].rename(columns={'press_rate':'r_t1'})
new_oc_t2 = pr.query("task==2 and block==2 and trial_role=='OC'")[['subject_id','press_rate']].rename(columns={'press_rate':'new_oc_t2'})
oc_df = former_oc.merge(oc_t1,'inner','subject_id').merge(r_t1,'inner','subject_id').merge(new_oc_t2,'left','subject_id')

print(f"\nG3.2 — OC property across tasks (N={len(oc_df)})")
print(f"  R baseline (T1):           {oc_df['r_t1'].mean():.3f}")
print(f"  OC in T1:                  {oc_df['oc_t1'].mean():.3f}")
print(f"  Former OC as R in T2:      {oc_df['former_oc_as_r_t2'].mean():.3f}")
print(f"  New OC in T2:              {oc_df['new_oc_t2'].mean():.3f}")
t_drop, p_drop = stats.ttest_rel(oc_df['former_oc_as_r_t2'], oc_df['oc_t1'])
t_base, p_base = stats.ttest_rel(oc_df['former_oc_as_r_t2'], oc_df['r_t1'])
print(f"  Former OC T2 vs OC T1:      t={t_drop:.3f}, p={p_drop:.4f}")
print(f"  Former OC T2 vs R baseline: t={t_base:.3f}, p={p_base:.4f}")

# G3.3: Transfer index distribution
sc_b2['transfer_idx'] = sc_b2['task2'] - sc_b2['task1']
ti = sc_b2['transfer_idx']
print(f"\nG3.3 — Transfer index (T2 SC − T1 SC, block 2, N={len(ti)})")
print(f"  Mean={ti.mean():.3f}  Median={ti.median():.3f}  Std={ti.std():.3f}")
print(f"  % positive: {(ti > 0).mean()*100:.1f}%   % negative: {(ti < 0).mean()*100:.1f}%")
t_ti, p_ti = stats.ttest_1samp(ti, 0)
print(f"  One-sample t vs 0: t={t_ti:.3f}, p={p_ti:.4f}")

# G3.4: Value-specific vs property-type transfer
sc_b2['value_type'] = sc_b2['sc_prop'].map(
    {'colour': 'value_changes', 'size': 'value_stable', 'velocity': 'value_stable'}
)
print(f"\nG3.4 — Transfer index by SC property")
print(sc_b2.groupby('sc_prop')['transfer_idx'].agg(n='count', mean='mean', sd='std').round(3))
t_vc, p_vc = stats.ttest_ind(
    sc_b2[sc_b2['sc_prop'] == 'colour']['transfer_idx'],
    sc_b2[sc_b2['sc_prop'] != 'colour']['transfer_idx']
)
print(f"  Colour vs size+velocity: t={t_vc:.3f}, p={p_vc:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
for prop, col, mk in [('colour','mediumorchid','o'),('size','steelblue','s'),('velocity','seagreen','^')]:
    d = sc_b2[sc_b2['sc_prop'] == prop]
    ax.scatter(d['task1'], d['task2'], color=col, marker=mk, alpha=0.6, label=prop, s=40)
ax.plot([0,1],[0,1],'k--',linewidth=1,label='no change')
ax.set(xlabel='Task 1 SC press rate', ylabel='Task 2 SC press rate',
       title='SC block 2: Task 1 vs Task 2', xlim=(-0.05,1.05), ylim=(-0.05,1.05))
ax.legend(fontsize=8)

ax = axes[1]
for vtype, col, lbl in [('value_changes','mediumorchid','colour (value changes)'),
                         ('value_stable','steelblue','size/velocity (stable)')]:
    d = sc_b2[sc_b2['value_type'] == vtype]['transfer_idx']
    ax.hist(d, bins=14, alpha=0.55, color=col, edgecolor='white', label=lbl)
ax.axvline(0, color='black', linestyle='--', linewidth=1.5)
ax.axvline(ti.mean(), color='red', linewidth=1.5, label=f'mean={ti.mean():.2f}')
ax.set(xlabel='Transfer index (T2 − T1)', ylabel='Subjects', title='Transfer index distribution')
ax.legend(fontsize=7)

ax = axes[2]
lbls  = ['R\n(T1 baseline)', 'OC\n(T1)', 'Former OC\nas R (T2)', 'New OC\n(T2)']
means = [oc_df['r_t1'].mean(), oc_df['oc_t1'].mean(), oc_df['former_oc_as_r_t2'].mean(), oc_df['new_oc_t2'].mean()]
sems  = [oc_df[c].sem() for c in ['r_t1','oc_t1','former_oc_as_r_t2','new_oc_t2']]
bars  = ax.bar(lbls, means, color=['grey','darkorange','darkorange','darkorange'],
               alpha=0.75, edgecolor='white', yerr=sems, capsize=4)
bars[2].set_alpha(0.35)
ax.axhline(oc_df['r_t1'].mean(), color='grey', linestyle='--', linewidth=1)
ax.set(ylabel='Mean press rate (±SEM)', title='OC property press rates across tasks', ylim=(0,1.05))

plt.tight_layout()
plt.savefig('../output/g3_transfer.png', dpi=150, bbox_inches='tight')


G3.1 — SC block 2 press rate (N=70)
  Task 1:  mean=0.501  sd=0.254
  Task 2:  mean=0.433  sd=0.273
  Paired t (T2 vs T1): t=-2.180, p=0.0327

  Early trials — first 10 of block 2 (N=70)
  Task 1 early: mean=0.454
  Task 2 early: mean=0.398
  Paired t: t=-1.180, p=0.2420

G3.2 — OC property across tasks (N=63)
  R baseline (T1):           0.408
  OC in T1:                  0.623
  Former OC as R in T2:      0.377
  New OC in T2:              0.572
  Former OC T2 vs OC T1:      t=-5.292, p=0.0000
  Former OC T2 vs R baseline: t=-0.832, p=0.4086

G3.3 — Transfer index (T2 SC − T1 SC, block 2, N=70)
  Mean=-0.067  Median=-0.049  Std=0.259
  % positive: 37.1%   % negative: 61.4%
  One-sample t vs 0: t=-2.180, p=0.0327

G3.4 — Transfer index by SC property
           n   mean     sd
sc_prop                   
colour    22 -0.082  0.277
size      25  0.016  0.239
velocity  23 -0.144  0.245
  Colour vs size+velocity: t=-0.324, p=0.7468


## GROUP 4 — Individual differences

- Between-subject variability in SC-R gap
- SC property type as moderator (colour vs size vs velocity)
- Group A vs B SC press rate distributions on same plot
- Block 3: is SC press rate stable, rising, or decaying per subject? Classify subjects into maintainers vs diffusers.


In [5]:
# G4.1: SC-R gap per subject
sc_r = (
    pr.query("block == 2 and trial_role in ['SC','R']")
    .pivot_table(index=['subject_id','group','sc_prop','task'], columns='trial_role', values='press_rate')
    .reset_index()
    .dropna(subset=['SC','R'])
)
sc_r['sc_r_gap'] = sc_r['SC'] - sc_r['R']

print("G4.1 — SC-R gap (block 2)")
print(sc_r.groupby('task')['sc_r_gap'].describe()[['count','mean','std','min','max']].round(3))

# G4.2: SC property type as moderator
print("\nG4.2 — SC-R gap by SC property type and task")
print(sc_r.groupby(['sc_prop','task'])['sc_r_gap'].agg(n='count', mean='mean', sd='std').round(3))
for task in [1, 2]:
    groups = [sc_r.query("sc_prop==@p and task==@task")['sc_r_gap'].dropna() for p in ['colour','size','velocity']]
    f, pf = stats.f_oneway(*groups)
    print(f"  One-way ANOVA task {task}: F={f:.3f}, p={pf:.4f}")

# G4.3: Group A vs B SC press rate
print("\nG4.3 — SC block 2 press rate by group and task")
grp = pr.query("block==2 and trial_role=='SC'")[['subject_id','group','task','press_rate']]
print(grp.groupby(['group','task'])['press_rate'].agg(n='count', mean='mean', sd='std').round(3))
for task in [1, 2]:
    a = grp.query("group=='A' and task==@task")['press_rate']
    b = grp.query("group=='B' and task==@task")['press_rate']
    t_g, p_g = stats.ttest_ind(a, b)
    print(f"  Group A vs B, task {task}: t={t_g:.3f}, p={p_g:.4f}")

# G4.4: Maintainers vs diffusers
b2_sc = pr.query("block==2 and trial_role=='SC'")[['subject_id','task','press_rate']].rename(columns={'press_rate':'b2'})
b3_sc = pr.query("block==3 and trial_role=='SC'")[['subject_id','task','press_rate']].rename(columns={'press_rate':'b3'})
stab  = b2_sc.merge(b3_sc, on=['subject_id','task']).dropna()
stab['change'] = stab['b3'] - stab['b2']

THR = 0.10
stab['cls'] = 'stable'
stab.loc[stab['change'] >  THR, 'cls'] = 'maintainer'
stab.loc[stab['change'] < -THR, 'cls'] = 'diffuser'

print(f"\nG4.4 — Block 3 SC stability (threshold ±{THR})")
print(stab.groupby(['task','cls']).size().unstack(fill_value=0))
print("\n  Mean B3−B2 change:")
print(stab.groupby('task')['change'].agg(mean='mean', sd='std', n='count').round(3))

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

ax = axes[0][0]
for task, col in [(1,'steelblue'),(2,'darkorange')]:
    d = sc_r[sc_r['task']==task]['sc_r_gap']
    ax.hist(d, bins=15, alpha=0.5, color=col, edgecolor='white', label=f'Task {task}')
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.set(xlabel='SC press rate − R press rate', ylabel='Subjects',
       title='SC-R gap distribution (block 2)')
ax.legend()

ax = axes[0][1]
x, w = np.arange(3), 0.35
for i, (task, col) in enumerate([(1,'steelblue'),(2,'darkorange')]):
    means = [sc_r.query("sc_prop==@p and task==@task")['sc_r_gap'].mean() for p in ['colour','size','velocity']]
    sems  = [sc_r.query("sc_prop==@p and task==@task")['sc_r_gap'].sem()  for p in ['colour','size','velocity']]
    ax.bar(x + i*w, means, w, color=col, alpha=0.75, label=f'Task {task}', yerr=sems, capsize=3)
ax.set(xticks=x+w/2, xticklabels=['colour','size','velocity'],
       ylabel='SC-R gap (±SEM)', title='SC-R gap by SC property type')
ax.axhline(0, color='black', linewidth=0.8)
ax.legend()

ax = axes[1][0]
for task, ls in [(1,'-'),(2,'--')]:
    for grp_val, col in [('A','steelblue'),('B','darkorange')]:
        d = pr.query("block==2 and trial_role=='SC' and group==@grp_val and task==@task")['press_rate']
        ax.hist(d, bins=12, alpha=0.25, color=col, edgecolor='white', histtype='stepfilled')
        ax.axvline(d.mean(), color=col, linestyle=ls, linewidth=2,
                   label=f'Grp {grp_val} T{task} ({d.mean():.2f})')
ax.set(xlabel='SC press rate (block 2)', ylabel='Subjects', title='Group A vs B SC press rate')
ax.legend(fontsize=7)

ax = axes[1][1]
for cls, col in [('maintainer','seagreen'),('stable','steelblue'),('diffuser','tomato')]:
    d = stab[stab['cls']==cls]
    ax.scatter(d['b2'], d['b3'], color=col, alpha=0.55, s=40, label=f'{cls} (n={len(d)})')
ax.plot([0,1],[0,1],'k--',linewidth=1)
ax.set(xlabel='Block 2 SC press rate', ylabel='Block 3 SC press rate',
       title='Block 3 stability (both tasks)', xlim=(-0.05,1.05), ylim=(-0.05,1.05))
ax.legend()

plt.tight_layout()
plt.savefig('../output/g4_individual_diffs.png', dpi=150, bbox_inches='tight')


G4.1 — SC-R gap (block 2)
      count   mean    std    min    max
task                                   
1      69.0  0.085  0.307 -1.000  0.808
2      66.0  0.049  0.273 -0.521  0.818

G4.2 — SC-R gap by SC property type and task
                n   mean     sd
sc_prop  task                  
colour   1     22  0.073  0.276
         2     21 -0.013  0.327
size     1     25  0.088  0.199
         2     25  0.132  0.200
velocity 1     22  0.094  0.428
         2     20  0.011  0.278
  One-way ANOVA task 1: F=0.027, p=0.9738
  One-way ANOVA task 2: F=1.926, p=0.1542

G4.3 — SC block 2 press rate by group and task
             n   mean     sd
group task                  
A     1     39  0.483  0.273
      2     39  0.381  0.263
B     1     31  0.522  0.231
      2     31  0.499  0.275
  Group A vs B, task 1: t=-0.633, p=0.5291
  Group A vs B, task 2: t=-1.827, p=0.0720

G4.4 — Block 3 SC stability (threshold ±0.1)
cls   diffuser  maintainer  stable
task                              
1   